In [ ]:
from google.colab import drive
import os
import json
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

rf_model = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase6_random_forest.joblib"
    )
)

print("Random Forest model loaded successfully.")
print(type(rf_model))

In [ ]:
with open(
    os.path.join(
        OUTPUT_PATH,
        "feature_inventory.json"
    ),
    "r"
) as f:
    feature_inventory = json.load(f)

current_features = feature_inventory[
    "current_features"
]

lag_features = feature_inventory[
    "lag_features"
]

rolling_features = feature_inventory[
    "rolling_features"
]

calendar_features = feature_inventory[
    "calendar_features"
]

event_features = feature_inventory[
    "event_features"
]

spatial_features = feature_inventory[
    "spatial_features"
]

target = feature_inventory[
    "target"
][0]

model_features = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
    + spatial_features
)

print("Current features:", len(current_features))
print("Lag features:", len(lag_features))
print("Rolling features:", len(rolling_features))
print("Calendar features:", len(calendar_features))
print("Event features:", len(event_features))
print("Spatial features:", len(spatial_features))
print("Total model features:", len(model_features))
print("Target:", target)

In [ ]:
historical_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase5_feature_engineered.parquet"
    )
)

historical_df["Datetime"] = pd.to_datetime(
    historical_df["Datetime"]
)

historical_df = historical_df.sort_values(
    [
        "StationId",
        "Datetime"
    ]
).reset_index(drop=True)

print("Historical dataset:", historical_df.shape)
print("Stations:", historical_df["StationId"].nunique())
print(
    "Range:",
    historical_df["Datetime"].min(),
    "→",
    historical_df["Datetime"].max()
)

In [ ]:
raw_live_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

required_live_columns = [
    "StationId",
    "Datetime"
] + raw_live_features

print("Required live input columns:")
print(required_live_columns)

In [ ]:
def assign_season(month):
    if month in [10, 11]:
        return "PostMonsoon"
    elif month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Other"

In [ ]:
diwali_dates = pd.to_datetime([
    "2015-11-11",
    "2016-10-30",
    "2017-10-19",
    "2018-11-07",
    "2019-10-27",
    "2020-11-14"
])

print(diwali_dates)

In [ ]:
def is_diwali_window(
    timestamp,
    days_before=2,
    days_after=2
):
    timestamp = pd.Timestamp(timestamp)

    return int(
        any(
            (
                timestamp
                >= date
                - pd.Timedelta(
                    days=days_before
                )
            )
            and
            (
                timestamp
                <= date
                + pd.Timedelta(
                    days=days_after
                )
            )
            for date in diwali_dates
        )
    )

In [ ]:
def build_live_features(
    history_df,
    station_id,
    prediction_time
):
    history_df = history_df.copy()

    history_df["Datetime"] = pd.to_datetime(
        history_df["Datetime"]
    )

    station_history = (
        history_df[
            history_df["StationId"].astype(str)
            == str(station_id)
        ]
        .sort_values("Datetime")
        .copy()
    )

    prediction_time = pd.Timestamp(
        prediction_time
    )

    # Only information available at t
    station_history = station_history[
        station_history["Datetime"]
        <= prediction_time
    ].copy()

    if len(station_history) == 0:
        raise ValueError(
            "No station history is available "
            "at or before the prediction time."
        )

    current_row = (
        station_history[
            station_history["Datetime"]
            == prediction_time
        ]
    )

    if len(current_row) == 0:
        raise ValueError(
            "The prediction timestamp does not "
            "exist in the supplied history."
        )

    current_row = current_row.iloc[-1]

    result = {
        "StationId": str(station_id),

        "AQI": current_row["AQI"],
        "PM2.5": current_row["PM2.5"],
        "PM10": current_row["PM10"],
        "NO": current_row["NO"],
        "NO2": current_row["NO2"],
        "NOx": current_row["NOx"],
        "NH3": current_row["NH3"],
        "CO": current_row["CO"],
        "SO2": current_row["SO2"],
        "O3": current_row["O3"],
        "Benzene": current_row["Benzene"],
        "Toluene": current_row["Toluene"],

        "Hour": prediction_time.hour,
        "Day": prediction_time.day,
        "Month": prediction_time.month,
        "Year": prediction_time.year,
        "DayOfWeek": prediction_time.dayofweek,
        "IsWeekend": int(
            prediction_time.dayofweek >= 5
        ),

        "IsStubbleSeason": int(
            prediction_time.month in [10, 11]
        ),

        "IsDiwaliWindow": is_diwali_window(
            prediction_time
        )
    }

    # --------------------------------------------------
    # Exact historical lags
    # --------------------------------------------------

    for lag in [1, 2, 12]:

        lag_time = (
            prediction_time
            - pd.Timedelta(
                hours=lag
            )
        )

        lag_row = station_history[
            station_history["Datetime"]
            == lag_time
        ]

        for column in [
            "AQI"
        ] + raw_live_features[1:]:

            feature_name = (
                f"{column}_lag_{lag}h"
            )

            if len(lag_row) == 0:
                result[feature_name] = np.nan
            else:
                result[feature_name] = (
                    lag_row.iloc[-1][column]
                )

    # --------------------------------------------------
    # Time-based rolling features
    # --------------------------------------------------

    rolling_history = (
        station_history[
            [
                "Datetime",
                "AQI",
                "PM2.5",
                "PM10",
                "NO2"
            ]
        ]
        .set_index("Datetime")
        .sort_index()
    )

    rolling_history = rolling_history[
        rolling_history.index
        <= prediction_time
    ]

    for column in [
        "AQI",
        "PM2.5",
        "PM10",
        "NO2"
    ]:

        result[
            f"{column}_rolling_mean_3h"
        ] = (
            rolling_history[column]
            .rolling(
                "3h",
                min_periods=1
            )
            .mean()
            .iloc[-1]
        )

        result[
            f"{column}_rolling_std_3h"
        ] = (
            rolling_history[column]
            .rolling(
                "3h",
                min_periods=2
            )
            .std()
            .iloc[-1]
        )

        result[
            f"{column}_rolling_mean_12h"
        ] = (
            rolling_history[column]
            .rolling(
                "12h",
                min_periods=1
            )
            .mean()
            .iloc[-1]
        )

        result[
            f"{column}_rolling_std_12h"
        ] = (
            rolling_history[column]
            .rolling(
                "12h",
                min_periods=2
            )
            .std()
            .iloc[-1]
        )

    return pd.DataFrame(
        [result]
    )

In [ ]:
test_station = (
    historical_df["StationId"]
    .astype(str)
    .iloc[0]
)

test_time = (
    historical_df[
        historical_df["StationId"].astype(str)
        == test_station
    ]["Datetime"]
    .max()
)

test_features = build_live_features(
    historical_df,
    test_station,
    test_time
)

missing_model_features = [
    column
    for column in model_features
    if column not in test_features.columns
]

extra_features = [
    column
    for column in test_features.columns
    if column not in model_features
    and column != "Datetime"
]

print(
    "Missing model features:",
    missing_model_features
)

print(
    "Extra generated features:",
    extra_features
)

print(
    "Feature count generated:",
    len(test_features.columns)
)

In [ ]:
# --------------------------------------------------
# Derive the exact feature contract from the
# saved Random Forest pipeline
# --------------------------------------------------

rf_preprocessor = rf_model.named_steps["preprocessor"]

trained_numeric_features = []
trained_categorical_features = []

for transformer_name, transformer, columns in (
    rf_preprocessor.transformers_
):

    if transformer_name == "numeric":
        trained_numeric_features = list(columns)

    elif transformer_name == "categorical":
        trained_categorical_features = list(columns)

model_features = (
    trained_numeric_features
    + trained_categorical_features
)

print("=" * 60)
print("RANDOM FOREST FEATURE CONTRACT")
print("=" * 60)

print(
    "Numeric features:",
    len(trained_numeric_features)
)

print(
    "Categorical features:",
    trained_categorical_features
)

print(
    "Total raw input features:",
    len(model_features)
)

print("\nFeatures:")
for feature in model_features:
    print("-", feature)

# --------------------------------------------------
# Check which expected features are missing
# --------------------------------------------------

missing_features = [
    feature
    for feature in model_features
    if feature not in test_features.columns
]

extra_features = [
    feature
    for feature in test_features.columns
    if feature not in model_features
]

print("\nMissing features:")
print(missing_features)

print("\nExtra generated features:")
print(extra_features)

In [ ]:
if missing_features:
    raise ValueError(
        "The live feature generator is missing "
        f"these model features: {missing_features}"
    )

test_features = test_features[
    model_features
].copy()

print(
    "Final inference shape:",
    test_features.shape
)

display(test_features)

In [ ]:
print(
    "Season included in inference:",
    "Season" in test_features.columns
)

print(
    "Feature count:",
    test_features.shape[1]
)

print(
    "Model feature count:",
    len(model_features)
)

In [ ]:
simulated_prediction = rf_model.predict(
    test_features
)

predicted_aqi_t_plus_1 = float(
    simulated_prediction[0]
)

print(
    "Station:",
    test_station
)

print(
    "Current time t:",
    test_time
)

print(
    "Predicted AQI at t+1:",
    round(
        predicted_aqi_t_plus_1,
        2
    )
)

In [ ]:
actual_target_row = historical_df[
    (
        historical_df["StationId"].astype(str)
        == test_station
    )
    &
    (
        historical_df["Datetime"]
        == (
            pd.Timestamp(test_time)
            + pd.Timedelta(hours=1)
        )
    )
]

if len(actual_target_row) == 1:

    actual_aqi_t_plus_1 = float(
        actual_target_row[
            "AQI_t_plus_1"
        ].iloc[0]
    )

    prediction_error = (
        actual_aqi_t_plus_1
        - predicted_aqi_t_plus_1
    )

    print(
        "Predicted AQI(t+1):",
        round(
            predicted_aqi_t_plus_1,
            2
        )
    )

    print(
        "Actual AQI(t+1):",
        round(
            actual_aqi_t_plus_1,
            2
        )
    )

    print(
        "Prediction error:",
        round(
            prediction_error,
            2
        )
    )

else:
    print(
        "Actual t+1 observation not available "
        "for this historical simulation."
    )

In [ ]:
simulation_records = []

rng = np.random.default_rng(42)

eligible_rows = (
    historical_df[
        historical_df["Datetime"]
        < historical_df["Datetime"].max()
    ]
    [
        [
            "StationId",
            "Datetime"
        ]
    ]
    .drop_duplicates()
)

sample_size = min(
    100,
    len(eligible_rows)
)

sample_indices = rng.choice(
    len(eligible_rows),
    size=sample_size,
    replace=False
)

for idx in sample_indices:

    station_id = str(
        eligible_rows.iloc[idx]["StationId"]
    )

    current_time = pd.Timestamp(
        eligible_rows.iloc[idx]["Datetime"]
    )

    try:

        live_features = build_live_features(
            historical_df,
            station_id,
            current_time
        )

        live_features = live_features[
            model_features
        ]

        prediction = float(
            rf_model.predict(
                live_features
            )[0]
        )

        actual_row = historical_df[
            (
                historical_df["StationId"].astype(str)
                == station_id
            )
            &
            (
                historical_df["Datetime"]
                == (
                    current_time
                    + pd.Timedelta(hours=1)
                )
            )
        ]

        if len(actual_row) == 1:

            actual = float(
                actual_row[
                    "AQI_t_plus_1"
                ].iloc[0]
            )

            simulation_records.append(
                {
                    "StationId": station_id,
                    "Datetime_t": current_time,
                    "Predicted_AQI_t_plus_1": prediction,
                    "Actual_AQI_t_plus_1": actual,
                    "AbsoluteError": abs(
                        actual - prediction
                    )
                }
            )

    except Exception:
        continue

simulation_results = pd.DataFrame(
    simulation_records
)

print(
    "Successful live-style simulations:",
    len(simulation_results)
)

display(
    simulation_results.head()
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

if len(simulation_results) > 0:

    simulated_mae = mean_absolute_error(
        simulation_results[
            "Actual_AQI_t_plus_1"
        ],
        simulation_results[
            "Predicted_AQI_t_plus_1"
        ]
    )

    simulated_rmse = np.sqrt(
        mean_squared_error(
            simulation_results[
                "Actual_AQI_t_plus_1"
            ],
            simulation_results[
                "Predicted_AQI_t_plus_1"
            ]
        )
    )

    simulated_r2 = r2_score(
        simulation_results[
            "Actual_AQI_t_plus_1"
        ],
        simulation_results[
            "Predicted_AQI_t_plus_1"
        ]
    )

    print(
        "Live-style simulation MAE:",
        round(
            simulated_mae,
            6
        )
    )

    print(
        "Live-style simulation RMSE:",
        round(
            simulated_rmse,
            6
        )
    )

    print(
        "Live-style simulation R²:",
        round(
            simulated_r2,
            6
        )
    )

else:
    print(
        "No successful simulations were produced."
    )

In [ ]:
inference_module = r'''
import numpy as np
import pandas as pd


def is_diwali_window(
    timestamp,
    diwali_dates,
    days_before=2,
    days_after=2
):
    timestamp = pd.Timestamp(timestamp)

    return int(
        any(
            (
                timestamp
                >= date
                - pd.Timedelta(
                    days=days_before
                )
            )
            and
            (
                timestamp
                <= date
                + pd.Timedelta(
                    days=days_after
                )
            )
            for date in diwali_dates
        )
    )


def build_live_features(
    history_df,
    station_id,
    prediction_time,
    diwali_dates
):
    prediction_time = pd.Timestamp(
        prediction_time
    )

    history_df = history_df.copy()

    history_df["Datetime"] = pd.to_datetime(
        history_df["Datetime"]
    )

    station_history = (
        history_df[
            history_df["StationId"].astype(str)
            == str(station_id)
        ]
        .sort_values("Datetime")
    )

    station_history = station_history[
        station_history["Datetime"]
        <= prediction_time
    ]

    current_rows = station_history[
        station_history["Datetime"]
        == prediction_time
    ]

    if len(current_rows) == 0:
        raise ValueError(
            "Current timestamp not found."
        )

    current = current_rows.iloc[-1]

    result = {
        "StationId": str(station_id),

        "AQI": current["AQI"],
        "PM2.5": current["PM2.5"],
        "PM10": current["PM10"],
        "NO": current["NO"],
        "NO2": current["NO2"],
        "NOx": current["NOx"],
        "NH3": current["NH3"],
        "CO": current["CO"],
        "SO2": current["SO2"],
        "O3": current["O3"],
        "Benzene": current["Benzene"],
        "Toluene": current["Toluene"],

        "Hour": prediction_time.hour,
        "Day": prediction_time.day,
        "Month": prediction_time.month,
        "Year": prediction_time.year,
        "DayOfWeek": prediction_time.dayofweek,
        "IsWeekend": int(
            prediction_time.dayofweek >= 5
        ),

        "IsStubbleSeason": int(
            prediction_time.month in [10, 11]
        ),

        "IsDiwaliWindow": is_diwali_window(
            prediction_time,
            diwali_dates
        )
    }

    for lag in [1, 2, 12]:

        lag_time = (
            prediction_time
            - pd.Timedelta(
                hours=lag
            )
        )

        lag_rows = station_history[
            station_history["Datetime"]
            == lag_time
        ]

        for column in [
            "AQI",
            "PM2.5",
            "PM10",
            "NO",
            "NO2",
            "NOx",
            "NH3",
            "CO",
            "SO2",
            "O3",
            "Benzene",
            "Toluene"
        ]:

            name = f"{column}_lag_{lag}h"

            if len(lag_rows) == 0:
                result[name] = np.nan
            else:
                result[name] = (
                    lag_rows.iloc[-1][column]
                )

    rolling_history = (
        station_history[
            [
                "Datetime",
                "AQI",
                "PM2.5",
                "PM10",
                "NO2"
            ]
        ]
        .set_index("Datetime")
        .sort_index()
    )

    for column in [
        "AQI",
        "PM2.5",
        "PM10",
        "NO2"
    ]:

        result[
            f"{column}_rolling_mean_3h"
        ] = (
            rolling_history[column]
            .rolling(
                "3h",
                min_periods=1
            )
            .mean()
            .iloc[-1]
        )

        result[
            f"{column}_rolling_std_3h"
        ] = (
            rolling_history[column]
            .rolling(
                "3h",
                min_periods=2
            )
            .std()
            .iloc[-1]
        )

        result[
            f"{column}_rolling_mean_12h"
        ] = (
            rolling_history[column]
            .rolling(
                "12h",
                min_periods=1
            )
            .mean()
            .iloc[-1]
        )

        result[
            f"{column}_rolling_std_12h"
        ] = (
            rolling_history[column]
            .rolling(
                "12h",
                min_periods=2
            )
            .std()
            .iloc[-1]
        )

    return pd.DataFrame([result])
'''

inference_module_path = os.path.join(
    OUTPUT_PATH,
    "live_inference.py"
)

with open(
    inference_module_path,
    "w"
) as f:
    f.write(
        inference_module
    )

print(
    "Inference module saved:"
)

print(
    inference_module_path
)

In [ ]:
simulation_path = os.path.join(
    TABLE_PATH,
    "phase12_live_style_simulation.csv"
)

simulation_results.to_csv(
    simulation_path,
    index=False
)

print(
    "Simulation results saved:"
)

print(
    simulation_path
)

In [ ]:
print("=" * 70)
print("PHASE 12 — LIVE-STYLE INFERENCE PIPELINE")
print("=" * 70)

print(
    "Model:",
    "Random Forest"
)

print(
    "Forecast:",
    "AQI(t) → AQI(t+1)"
)

print(
    "CPCB/OGD API connected:",
    False
)

print(
    "Successful historical simulations:",
    len(simulation_results)
)

if len(simulation_results) > 0:

    print(
        "Simulation MAE:",
        round(
            simulated_mae,
            6
        )
    )

    print(
        "Simulation RMSE:",
        round(
            simulated_rmse,
            6
        )
    )

    print(
        "Simulation R²:",
        round(
            simulated_r2,
            6
        )
    )

print(
    "\nNext:",
    "Phase 13 - CPCB/OGD Live Data Integration"
)

print("=" * 70)

In [ ]:
import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save historical live-style simulation results
# --------------------------------------------------

simulation_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase12_live_style_simulation.csv"
    ),
    index=False
)

# --------------------------------------------------
# 2. Save deployment configuration
# --------------------------------------------------

deployment_config = {
    "model": "Random Forest",
    "model_file": "phase6_random_forest.joblib",

    "forecast_target": "AQI_t_plus_1",
    "forecast_horizon_hours": 1,

    "input_available_until": "t",

    "cpcb_ogd_api_connected": False,

    "historical_simulations": int(
        len(simulation_results)
    ),

    "simulation_mae": float(
        simulated_mae
    ),

    "simulation_rmse": float(
        simulated_rmse
    ),

    "simulation_r2": float(
        simulated_r2
    ),

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
    )
}

deployment_path = os.path.join(
    OUTPUT_PATH,
    "phase12_deployment_config.json"
)

with open(
    deployment_path,
    "w"
) as f:
    json.dump(
        deployment_config,
        f,
        indent=4
    )

# --------------------------------------------------
# 3. Save Phase 12 summary
# --------------------------------------------------

phase12_summary = {
    "completed_phase": (
        "Phase 12 - Live-Style Inference Pipeline"
    ),

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
    ),

    "current_model": "Random Forest",

    "forecast": (
        "Current time t -> next-hour AQI at t+1"
    ),

    "historical_simulations": int(
        len(simulation_results)
    ),

    "simulation_mae": float(
        simulated_mae
    ),

    "simulation_rmse": float(
        simulated_rmse
    ),

    "simulation_r2": float(
        simulated_r2
    ),

    "api_connected": False
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase12_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        phase12_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 4. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": (
        "Phase 12 - Live-Style Inference Pipeline"
    ),

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
    ),

    "current_model": "Random Forest",

    "api_connected": False
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 5. Verify checkpoint
# --------------------------------------------------

files_to_check = [
    os.path.join(
        TABLE_PATH,
        "phase12_live_style_simulation.csv"
    ),
    deployment_path,
    summary_path,
    status_path
]

print("=" * 60)
print("PHASE 12 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nPhase 12 completed.")
print("Current model:", "Random Forest")
print("Next: Phase 13 - CPCB/OGD Live Data Integration")